# OIBSIP – Level 1 Task 1: Exploratory Data Analysis on Online Retail

**Dataset:** UCI Machine Learning Repository – Online Retail  
**Source file:** `data/Online Retail.xlsx`  
**Coverage:** 1 December 2010 to 9 December 2011  
**Important:** This is genuine historical transaction data supplied by UCI, not synthetic/demo data. Currency is treated as GBP based on the dataset documentation. The source contains no customer age, gender, or product-category field, so those requested cuts cannot be honestly calculated.

## Objective
Clean and explore transaction-level retail data, quantify revenue trends, identify leading products and markets, and provide evidence-based business recommendations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("data/Online Retail.xlsx")
raw = pd.read_excel(DATA_PATH)
print("Raw shape:", raw.shape)
display(raw.head())
display(raw.dtypes)
display(raw.isna().sum().rename("missing_values"))


## 1. Data cleaning

Rules used:
- Remove cancelled invoices (invoice number starts with `C`).
- Remove rows with zero/negative quantity or unit price.
- Remove rows with missing product descriptions.
- Retain rows with missing CustomerID for overall revenue calculations; exclude them only from customer-level analysis.
- Calculate line revenue as `Quantity × UnitPrice`.

These are analytical filters, not edits to the original source file.


In [ ]:
df = raw.copy()
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["Description"] = df["Description"].astype("string").str.strip()
df["Country"] = df["Country"].astype("string").str.strip()
df["IsCancelled"] = df["InvoiceNo"].str.startswith("C", na=False)

clean = df.loc[
    (~df["IsCancelled"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0) &
    (df["Description"].notna())
].copy()

clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]
clean["Month"] = clean["InvoiceDate"].dt.to_period("M").astype(str)
clean["Quarter"] = clean["InvoiceDate"].dt.to_period("Q").astype(str)
clean["YearMonth"] = clean["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
clean["Hour"] = clean["InvoiceDate"].dt.hour
clean["Weekday"] = clean["InvoiceDate"].dt.day_name()

print("Raw rows:", len(raw))
print("Cancelled invoices:", int(df["IsCancelled"].sum()))
print("Rows retained for sales analysis:", len(clean))
print("Date range:", clean["InvoiceDate"].min(), "to", clean["InvoiceDate"].max())
print("Distinct invoices:", clean["InvoiceNo"].nunique())
print("Distinct customers (known IDs):", clean["CustomerID"].nunique())
print("Countries:", clean["Country"].nunique())


## 2. Descriptive statistics

In [ ]:
display(clean[["Quantity", "UnitPrice", "Revenue"]].describe().T)
for col in ["Quantity", "UnitPrice", "Revenue"]:
    print(f"{col}: mean={clean[col].mean():.2f}, median={clean[col].median():.2f}, mode={clean[col].mode().iloc[0]:.2f}, std={clean[col].std():.2f}")


## 3. Monthly and quarterly revenue trends

In [ ]:
monthly = clean.groupby("Month")["Revenue"].sum().sort_index()
quarterly = clean.groupby("Quarter")["Revenue"].sum().sort_index()

monthly.plot(figsize=(11,5), marker="o", title="Monthly Revenue Trend")
plt.ylabel("Revenue (GBP)"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

quarterly.plot(kind="bar", figsize=(9,5), title="Quarterly Revenue")
plt.ylabel("Revenue (GBP)"); plt.tight_layout(); plt.show()

display(monthly.sort_values(ascending=False).head(5))
display(quarterly)


## 4. Best-selling products and country revenue

In [ ]:
product_summary = (clean.groupby("Description")
    .agg(Units_Sold=("Quantity","sum"), Revenue=("Revenue","sum"))
    .sort_values("Units_Sold", ascending=False))
display(product_summary.head(10))

country_summary = clean.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
display(country_summary.head(10))

product_summary.head(10)["Units_Sold"].sort_values().plot(kind="barh", figsize=(10,6), title="Top 10 Products by Units Sold")
plt.xlabel("Units sold"); plt.tight_layout(); plt.show()

country_summary.head(10).sort_values().plot(kind="barh", figsize=(10,6), title="Top 10 Countries by Revenue")
plt.xlabel("Revenue (GBP)"); plt.tight_layout(); plt.show()


## 5. Customer analysis

CustomerID is missing on a portion of transactions. The customer-level ranking below uses only rows with a known CustomerID. Age and gender are not present in the source and are therefore not inferred.


In [ ]:
known_customers = clean.dropna(subset=["CustomerID"])
customer_summary = known_customers.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False)
print("Rows with missing CustomerID:", int(clean["CustomerID"].isna().sum()))
print("Share missing CustomerID:", f"{clean['CustomerID'].isna().mean():.1%}")
display(customer_summary.head(10))


## 6. Correlation and additional analysis

In [ ]:
corr = clean[["Quantity","UnitPrice","Revenue"]].corr()
display(corr)
plt.figure(figsize=(7,5))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=25)
plt.yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")
plt.title("Correlation Heatmap"); plt.colorbar(); plt.tight_layout(); plt.show()

weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday_sales = clean.groupby("Weekday")["Revenue"].sum().reindex(weekday_order)
weekday_sales.plot(kind="bar", figsize=(9,5), title="Revenue by Weekday")
plt.ylabel("Revenue (GBP)"); plt.tight_layout(); plt.show()


## 7. Findings and recommendations

The figures should be interpreted as **line-item revenue** calculated from quantity × unit price. They are not audited financial statements or profit.

1. The United Kingdom is the dominant market in this dataset; consider protecting availability and customer experience there while testing growth in other markets.
2. November 2011 is the strongest month in the observed period; plan inventory and fulfilment capacity ahead of seasonal peaks.
3. The leading products by units sold are useful candidates for stock availability monitoring and bundle testing.
4. CustomerID is missing for a meaningful share of retained rows; improve customer identification at checkout to strengthen retention and lifetime-value analysis.
5. Review cancellation/return patterns separately before making margin or product-quality decisions; this sales-only cleaning intentionally excludes cancelled and non-positive lines.

## Limitations
- No customer age or gender fields.
- No product category field.
- No cost, discount, shipping, or profit fields; revenue is not profit.
- Dataset covers a historical period ending in December 2011.
- Customer-level analysis excludes records without CustomerID.
